# EHRSHOT Benchmark: CLMBR-T-base vs TrajGPT

Analysis of few-shot evaluation results across 15 clinical prediction tasks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Load results
results_dir = Path('../results')

dfs = []
for model_dir in results_dir.iterdir():
    if not model_dir.is_dir():
        continue
    summary_path = model_dir / 'summary.csv'
    if summary_path.exists():
        dfs.append(pd.read_csv(summary_path))

df = pd.concat(dfs, ignore_index=True)
print(f'Models: {df["model"].unique()}')
print(f'Tasks: {df["task"].nunique()}')
print(f'Rows: {len(df)}')
df.head()

## AUROC vs k-shot (per task)

In [ ]:
auroc = df[df['metric'] == 'auroc']
tasks = sorted(auroc['task'].unique())
models = sorted(auroc['model'].unique())

fig, axes = plt.subplots(3, 5, figsize=(24, 14), sharey=False)
axes = axes.flatten()

for i, task in enumerate(tasks):
    ax = axes[i]
    for model in models:
        subset = auroc[(auroc['task'] == task) & (auroc['model'] == model)].sort_values('k')
        if len(subset) == 0:
            continue
        ax.plot(subset['k'], subset['mean'], 'o-', label=model, markersize=4)
        ax.fill_between(subset['k'], subset['ci_lower'], subset['ci_upper'], alpha=0.15)
    ax.set_title(task, fontsize=10)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('k')
    if i % 5 == 0:
        ax.set_ylabel('AUROC')
    if i == 0:
        ax.legend(fontsize=8)

# Hide unused axes
for j in range(len(tasks), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('AUROC vs k-shot per Task', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/comparison/auroc_per_task.png', dpi=150, bbox_inches='tight')
plt.show()

## Macro AUROC by Category

In [ ]:
categories = ['operational', 'lab', 'diagnosis', 'imaging']

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

for i, cat in enumerate(categories):
    ax = axes[i]
    cat_df = auroc[auroc['category'] == cat]
    for model in models:
        subset = cat_df[cat_df['model'] == model].groupby('k')['mean'].mean().reset_index()
        if len(subset) == 0:
            continue
        ax.plot(subset['k'], subset['mean'], 'o-', label=model, markersize=5)
    ax.set_title(cat.title())
    ax.set_xscale('log', base=2)
    ax.set_xlabel('k-shot')
    if i == 0:
        ax.set_ylabel('Macro AUROC')
        ax.legend()

plt.suptitle('Macro AUROC by Task Category', fontsize=14)
plt.tight_layout()
plt.savefig('../results/comparison/macro_auroc_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Table (k=128)

In [ ]:
# Pivot table at k=128
k128 = auroc[auroc['k'] == 128].pivot_table(
    index='task', columns='model', values='mean'
).round(4)

k128['diff'] = k128.iloc[:, -1] - k128.iloc[:, 0] if k128.shape[1] >= 2 else 0
k128 = k128.sort_values('diff', ascending=False)
print(k128.to_string())
k128